# Etapa 2 — Preparação dos dados

Objetivo: transformar as planilhas em matrizes prontas para a rede.

## Recarregando e limpando a base

Repete o pipeline de limpeza decidido na Etapa 1 (`01_exploracao.ipynb` e
`DIARIO.md`): normalizar a chave de junção, ficar só com o recorte de 8
espécies, descartar as 5 gravações confirmadas como erro grave.

In [1]:
import re
import numpy as np
import pandas as pd

np.random.seed(42)

BANCO = "../banco/Cidade Viva e PELG Bioacústica (OFICIAL).xlsx - {}.csv"

In [2]:
ident_cv = pd.read_csv(BANCO.format("Cidade Viva - Identificação"))
ident_pelg = pd.read_csv(BANCO.format("PELG - Identificação"))
identificacao = pd.concat([ident_cv, ident_pelg], ignore_index=True)

valores_cv = pd.read_csv(BANCO.format("Cidade Viva - Valores"))
valores_pelg = pd.read_csv(BANCO.format("PELG - Valores"))
valores = pd.concat([valores_cv, valores_pelg], ignore_index=True)

print("Identificação:", identificacao.shape, " Valores (bruto, com linhas vazias):", valores.shape)

Identificação: (14540, 41)  Valores (bruto, com linhas vazias): (44757, 15)


In [3]:
def normaliza_arquivo(nome):
    if pd.isna(nome):
        return nome
    nome = nome.strip()
    nome = nome.split("(")[0].strip()
    nome = re.sub(r"_000$", "", nome)
    return nome


# aba Valores tem colunas numéricas com vírgula decimal em algumas linhas (Etapa 1)
# — converter ANTES de decidir o que é "linha vazia"
colunas_numericas_valores = ["Fstart(s)", "Fend(s)", "Fmin(kHz)", "Fmax(kHz)", "Bandwidth (kHz)",
                              "FME(kHz)", "Dur(ms)"]
for col in colunas_numericas_valores:
    valores[col] = pd.to_numeric(
        valores[col].astype(str).str.replace(",", ".", regex=False), errors="coerce"
    )

# linhas onde nenhuma medida de pulso existe são espaço vazio sobrando na
# planilha, não um pulso de verdade — descartar ANTES do ffill, senão o ffill
# gruda essas linhas fantasmas na gravação anterior (foi o que gerou o grupo
# de "951 pulsos" que investigamos acima)
colunas_medida = ["Fmin(kHz)", "Fmax(kHz)", "FME(kHz)", "Dur(ms)"]
linha_vazia = valores[colunas_medida].isna().all(axis=1)
print(f"Linhas totalmente vazias na aba Valores: {linha_vazia.sum()} de {len(valores)}")
valores = valores[~linha_vazia].copy()

valores["ARQUIVO"] = valores["ARQUIVO"].ffill()

identificacao["Arquivo_norm"] = identificacao["Arquivo"].apply(normaliza_arquivo)
valores["Arquivo_norm"] = valores["ARQUIVO"].apply(normaliza_arquivo)

print("Valores (sem linhas vazias):", valores.shape)

Linhas totalmente vazias na aba Valores: 1260 de 44757


Valores (sem linhas vazias): (43497, 16)


In [4]:
# recorte de 8 espécies (Etapa 1): >= 379 gravações, excluindo o sonótipo Molossidae1
contagem_especies = identificacao["Espécie/Sonótipo"].value_counts().drop("Molossidae1")
especies_recorte = contagem_especies[contagem_especies >= 379].index.tolist()

ident_recorte = identificacao[identificacao["Espécie/Sonótipo"].isin(especies_recorte)].copy()

# 5 gravações confirmadas como erro grave (DIARIO.md, Etapa 1)
col_fmin, col_fmax, col_fme, col_dur = "Fmin (kHz)", "Fmax (kHz)", "FME (kHz)", "Duração (ms)"

linha_lavali = ident_recorte[(ident_recorte["Espécie/Sonótipo"] == "Myotis_lavali") & (ident_recorte[col_fme] < 10)]
linha_eumops = ident_recorte[(ident_recorte["Espécie/Sonótipo"] == "Eumops_sp") & (ident_recorte[col_fme] > 45)]
ident_recorte = ident_recorte.drop(index=pd.concat([linha_lavali, linha_eumops]).index)

# duplicata EXATA (mesmo arquivo, mesmo rótulo) = erro de digitação, manter 1 cópia.
# Arquivo_norm repetido com rótulos DIFERENTES não é erro: é gravação com mais de
# uma espécie, e a planilha corretamente tem uma linha por espécie presente — não
# mexe na representação simples, só vira um problema na completa (nível de pulso),
# tratado separadamente mais abaixo.
antes = len(ident_recorte)
ident_recorte = ident_recorte.drop_duplicates(subset=["Arquivo_norm", "Espécie/Sonótipo"], keep="first")

print("Espécies no recorte:", len(especies_recorte))
print("Duplicatas exatas removidas:", antes - len(ident_recorte))
print("Gravações no recorte, após limpeza:", len(ident_recorte))

Espécies no recorte: 8
Duplicatas exatas removidas: 27
Gravações no recorte, após limpeza: 13408


## Representação simples

Os 5 descritores médios da aba Identificação. `Bandwidth` é recalculado como
`Fmax - Fmin` (Etapa 1: a coluna original tem valores corrompidos numa fração
dos casos). É com essa representação que as Etapas A e B começam.

In [5]:
X_simples = np.column_stack([
    ident_recorte[col_fmin].to_numpy(),
    ident_recorte[col_fmax].to_numpy(),
    (ident_recorte[col_fmax] - ident_recorte[col_fmin]).to_numpy(),  # Bandwidth recalculado
    ident_recorte[col_fme].to_numpy(),
    ident_recorte[col_dur].to_numpy(),
])
colunas_simples = ["Fmin", "Fmax", "Bandwidth", "FME", "Duração"]
y_simples = ident_recorte["Espécie/Sonótipo"].to_numpy()
arquivos_simples = ident_recorte["Arquivo_norm"].to_numpy()

print("X_simples:", X_simples.shape)
print("Colunas:", colunas_simples)
print("Algum NaN?", np.isnan(X_simples).any())

X_simples: (13408, 5)
Colunas: ['Fmin', 'Fmax', 'Bandwidth', 'FME', 'Duração']
Algum NaN? False


## Representação completa

Antes de montar, preciso confirmar quantos pulsos cada gravação tem na aba
Valores — o plano diz "3 pulsos individuais", mas a coluna `N_pulsos` da
Identificação mostrava valores bem maiores (8, 9...) em alguns exemplos que
vimos na Etapa 1. Confirmando qual é a contagem real na aba Valores antes de
fixar o formato da matriz.

In [6]:
arquivos_recorte_norm = set(ident_recorte["Arquivo_norm"])
valores_recorte = valores[valores["Arquivo_norm"].isin(arquivos_recorte_norm)].copy()

pulsos_por_gravacao = valores_recorte.groupby("Arquivo_norm").size()
print("Pulsos por gravação — distribuição:")
print(pulsos_por_gravacao.value_counts().sort_index())

Pulsos por gravação — distribuição:
3    12873
6      316
9        9
Name: count, dtype: int64


In [7]:
suspeito = pulsos_por_gravacao.idxmax()
print("Arquivo_norm com mais pulsos:", suspeito, "->", pulsos_por_gravacao[suspeito])
print("\nValores originais de ARQUIVO que caem nesse grupo:")
print(valores_recorte.loc[valores_recorte["Arquivo_norm"] == suspeito, "ARQUIVO"].unique()[:10])

In [8]:
grupo = valores_recorte[valores_recorte["Arquivo_norm"] == suspeito]
print(grupo[["Fmin(kHz)", "Fmax(kHz)", "FME(kHz)", "Dur(ms)"]].describe())

       Fmin(kHz)  Fmax(kHz)   FME(kHz)    Dur(ms)
count   9.000000   9.000000   9.000000   9.000000
mean   40.252667  46.063000  39.492667   3.883778
std     9.611721  12.693135  11.110053   2.477624
min    24.922000  27.028000  25.587000   2.289000
25%    38.240000  42.360000  25.888000   2.481000
50%    40.421000  47.065000  40.956000   3.075000
75%    49.691000  56.740000  50.666000   3.761000
max    50.641000  62.638000  51.430000  10.249000

Confirmado: são gravações com mais de uma espécie (`Tem_outra_espécie`). Como a
Identificação dá um único rótulo por gravação, usar os pulsos dessas gravações
misturaria pulsos de espécies diferentes sob um só rótulo. Decisão: manter só
as gravações com exatamente 3 pulsos (inequívocas) na representação completa.

In [9]:
especies_por_arquivo = ident_recorte.groupby("Arquivo_norm")["Espécie/Sonótipo"].nunique()
arquivos_especie_unica = especies_por_arquivo[especies_por_arquivo == 1].index

gravacoes_3pulsos = pulsos_por_gravacao[pulsos_por_gravacao == 3].index
gravacoes_completa = set(gravacoes_3pulsos) & set(arquivos_especie_unica)

valores_completa = valores_recorte[valores_recorte["Arquivo_norm"].isin(gravacoes_completa)].copy()
ident_completa = ident_recorte[ident_recorte["Arquivo_norm"].isin(gravacoes_completa)].copy()

print(f"Gravações na representação completa: {len(ident_completa)} de {len(ident_recorte)} "
      f"({len(ident_completa)/len(ident_recorte)*100:.1f}%)")
print(f"Pulsos: {len(valores_completa)} ({len(valores_completa)/3:.0f} gravações x 3)")

Gravações na representação completa: 12871 de 13408 (96.0%)
Pulsos: 38613 (12871 gravações x 3)


Montando a matriz: para cada gravação, os 3 pulsos individuais (Fmin, Fmax,
Bandwidth recalculado, FME, Duração — 5 × 3 = 15 colunas) + média e
desvio-padrão de cada descritor entre os 3 pulsos (5 × 2 = 10 colunas) + tipo
de pulso em one-hot (4 colunas, da Identificação). Fstart/Fend ficam de fora
(Etapa 1: artefato de software). Total: 29 colunas.

In [10]:
valores_completa = valores_completa.sort_values(["Arquivo_norm"], kind="stable").copy()
valores_completa["Bandwidth_recalc"] = valores_completa["Fmax(kHz)"] - valores_completa["Fmin(kHz)"]
valores_completa["ordem_pulso"] = valores_completa.groupby("Arquivo_norm").cumcount()

cols_pulso = ["Fmin(kHz)", "Fmax(kHz)", "Bandwidth_recalc", "FME(kHz)", "Dur(ms)"]

# uma linha por gravação, uma coluna por (pulso, descritor)
pulsos_wide = valores_completa.pivot(index="Arquivo_norm", columns="ordem_pulso", values=cols_pulso)
pulsos_wide.columns = [f"{col}_pulso{p+1}" for col, p in pulsos_wide.columns]

# média e desvio-padrão entre os 3 pulsos, por descritor
media_pulsos = valores_completa.groupby("Arquivo_norm")[cols_pulso].mean()
media_pulsos.columns = [f"{c}_media" for c in media_pulsos.columns]
std_pulsos = valores_completa.groupby("Arquivo_norm")[cols_pulso].std()
std_pulsos.columns = [f"{c}_std" for c in std_pulsos.columns]

print("pulsos_wide:", pulsos_wide.shape)
print("media_pulsos:", media_pulsos.shape, " std_pulsos:", std_pulsos.shape)

pulsos_wide: (12871, 15)
media_pulsos: (12871, 5)  std_pulsos: (12871, 5)


In [11]:
# one-hot de Tipo_pulso (nível gravação, vem da Identificação)
tipo_pulso_onehot = pd.get_dummies(
    ident_completa.set_index("Arquivo_norm")["Tipo_pulso"], prefix="tipo"
).astype(float)

completa = pulsos_wide.join(media_pulsos).join(std_pulsos).join(tipo_pulso_onehot)

# alinhar rótulos na mesma ordem do índice de "completa"
rotulo_por_arquivo = ident_completa.set_index("Arquivo_norm")["Espécie/Sonótipo"]
y_completa = rotulo_por_arquivo.loc[completa.index].to_numpy()

X_completa = completa.to_numpy()
colunas_completa = completa.columns.tolist()

print("X_completa:", X_completa.shape)
print("Algum NaN?", np.isnan(X_completa).any())
print("\nColunas:", colunas_completa)

X_completa: (12871, 29)
Algum NaN? True

Colunas: ['Fmin(kHz)_pulso1', 'Fmin(kHz)_pulso2', 'Fmin(kHz)_pulso3', 'Fmax(kHz)_pulso1', 'Fmax(kHz)_pulso2', 'Fmax(kHz)_pulso3', 'Bandwidth_recalc_pulso1', 'Bandwidth_recalc_pulso2', 'Bandwidth_recalc_pulso3', 'FME(kHz)_pulso1', 'FME(kHz)_pulso2', 'FME(kHz)_pulso3', 'Dur(ms)_pulso1', 'Dur(ms)_pulso2', 'Dur(ms)_pulso3', 'Fmin(kHz)_media', 'Fmax(kHz)_media', 'Bandwidth_recalc_media', 'FME(kHz)_media', 'Dur(ms)_media', 'Fmin(kHz)_std', 'Fmax(kHz)_std', 'Bandwidth_recalc_std', 'FME(kHz)_std', 'Dur(ms)_std', 'tipo_FMa/qCF', 'tipo_FMa/qCFd', 'tipo_FMd/qCFd', 'tipo_qCFd']


In [12]:
nan_por_coluna = completa.isna().sum()
print(nan_por_coluna[nan_por_coluna > 0])

FME(kHz)_pulso3    2
Dur(ms)_pulso1     1
Dur(ms)_pulso2     2
dtype: int64


In [13]:
# resíduo pequeno (5 valores em ~373k células) — descartar as gravações afetadas
antes = len(completa)
completa = completa.dropna()
print(f"Gravações descartadas por NaN residual: {antes - len(completa)}")

rotulo_por_arquivo = ident_completa.set_index("Arquivo_norm")["Espécie/Sonótipo"]
y_completa = rotulo_por_arquivo.loc[completa.index].to_numpy()
X_completa = completa.to_numpy()
colunas_completa = completa.columns.tolist()

print("X_completa final:", X_completa.shape)
print("Algum NaN?", np.isnan(X_completa).any())

Gravações descartadas por NaN residual: 5


X_completa final: (12866, 29)
Algum NaN? False


## Confirmando exclusão de variáveis de contexto

Área, estação e hora deixariam a rede aprender a identificar a espécie pela
fauna do local/época em vez do chamado acústico em si — por isso nunca
entraram nas colunas usadas. Confirmando que nenhuma das duas representações
inclui esse tipo de variável.

In [14]:
contexto = {"área", "estação", "hora", "município", "estado", "país", "latitude", "longitude", "data"}

print("Colunas simples:", colunas_simples)
print("Cruzam com contexto?", any(c.lower() in contexto for c in colunas_simples))
print()
print("Colunas completa:", colunas_completa)
print("Cruzam com contexto?", any(c.lower() in contexto for c in colunas_completa))

Colunas simples: ['Fmin', 'Fmax', 'Bandwidth', 'FME', 'Duração']
Cruzam com contexto? False

Colunas completa: ['Fmin(kHz)_pulso1', 'Fmin(kHz)_pulso2', 'Fmin(kHz)_pulso3', 'Fmax(kHz)_pulso1', 'Fmax(kHz)_pulso2', 'Fmax(kHz)_pulso3', 'Bandwidth_recalc_pulso1', 'Bandwidth_recalc_pulso2', 'Bandwidth_recalc_pulso3', 'FME(kHz)_pulso1', 'FME(kHz)_pulso2', 'FME(kHz)_pulso3', 'Dur(ms)_pulso1', 'Dur(ms)_pulso2', 'Dur(ms)_pulso3', 'Fmin(kHz)_media', 'Fmax(kHz)_media', 'Bandwidth_recalc_media', 'FME(kHz)_media', 'Dur(ms)_media', 'Fmin(kHz)_std', 'Fmax(kHz)_std', 'Bandwidth_recalc_std', 'FME(kHz)_std', 'Dur(ms)_std', 'tipo_FMa/qCF', 'tipo_FMa/qCFd', 'tipo_FMd/qCFd', 'tipo_qCFd']
Cruzam com contexto? False


## Divisão estratificada treino / validação / teste (70/15/15)

Implementada com NumPy: para cada espécie separadamente, embaralha os índices
(semente fixa) e corta 70/15/15. Feita em cima de `ident_recorte` (todas as
13.408 gravações da representação simples) — como `ident_completa` preserva os
índices originais de `ident_recorte` (é um subconjunto, não uma cópia
reindexada), a mesma divisão vale automaticamente para as duas representações.

In [15]:
def divisao_estratificada(rotulos, indices, proporcoes=(0.70, 0.15, 0.15), semente=42):
    """Para cada classe em `rotulos`, embaralha seus índices e corta nas
    proporções dadas. Retorna três arrays de índices (treino, val, teste)."""
    assert abs(sum(proporcoes) - 1.0) < 1e-9

    rng = np.random.default_rng(semente)
    treino, val, teste = [], [], []

    for classe in np.unique(rotulos):
        idx_classe = indices[rotulos == classe]
        idx_classe = rng.permutation(idx_classe)

        n = len(idx_classe)
        n_treino = int(round(n * proporcoes[0]))
        n_val = int(round(n * proporcoes[1]))
        # o resto vai pro teste, evita que arredondamento perca/sobre gravações

        treino.append(idx_classe[:n_treino])
        val.append(idx_classe[n_treino:n_treino + n_val])
        teste.append(idx_classe[n_treino + n_val:])

    return np.concatenate(treino), np.concatenate(val), np.concatenate(teste)


rotulos_split = ident_recorte["Espécie/Sonótipo"].to_numpy()
indices_split = ident_recorte.index.to_numpy()

idx_treino, idx_val, idx_teste = divisao_estratificada(rotulos_split, indices_split)

print(f"Treino: {len(idx_treino)} ({len(idx_treino)/len(indices_split)*100:.1f}%)")
print(f"Validação: {len(idx_val)} ({len(idx_val)/len(indices_split)*100:.1f}%)")
print(f"Teste: {len(idx_teste)} ({len(idx_teste)/len(indices_split)*100:.1f}%)")
print(f"Total: {len(idx_treino) + len(idx_val) + len(idx_teste)} (esperado {len(indices_split)})")

Treino: 9386 (70.0%)
Validação: 2011 (15.0%)
Teste: 2011 (15.0%)
Total: 13408 (esperado 13408)


In [16]:
# conferindo a proporção por espécie, não só no total
conjuntos = {"treino": idx_treino, "validação": idx_val, "teste": idx_teste}
tabela = pd.DataFrame({
    nome: ident_recorte.loc[idx, "Espécie/Sonótipo"].value_counts()
    for nome, idx in conjuntos.items()
})
tabela["total"] = tabela.sum(axis=1)
tabela["%treino"] = (tabela["treino"] / tabela["total"] * 100).round(1)
tabela["%val"] = (tabela["validação"] / tabela["total"] * 100).round(1)
tabela["%teste"] = (tabela["teste"] / tabela["total"] * 100).round(1)
tabela

,treino,validação,teste,total,%treino,%val,%teste
Espécie/Sonótipo,,,,,,,
Myotis_lavali,5099,1093,1092,7284,70.0,15.0,15.0
Molossus_molossus,1154,247,247,1648,70.0,15.0,15.0
Lasiurus_blossevillii,757,162,163,1082,70.0,15.0,15.1
Neoeptesicus_furinalis,710,152,152,1014,70.0,15.0,15.0
Eumops_sp,506,108,109,723,70.0,14.9,15.1
Molossus_currentium,501,107,108,716,70.0,14.9,15.1
Molossops_temminckii,390,84,83,557,70.0,15.1,14.9
Myotis_nigricans,269,58,57,384,70.1,15.1,14.8


## Levando a divisão para a representação completa

`idx_treino/val/teste` são índices de linha de `ident_recorte`. A matriz
`completa` é indexada por `Arquivo_norm`, não pelo índice numérico — preciso
traduzir antes de aplicar a mesma divisão lá.

In [17]:
# split, por índice de ident_recorte
split_por_indice = pd.Series(index=ident_recorte.index, dtype=object)
split_por_indice.loc[idx_treino] = "treino"
split_por_indice.loc[idx_val] = "validação"
split_por_indice.loc[idx_teste] = "teste"

# traduz para Arquivo_norm via ident_completa (single-species, índice original preservado)
split_por_arquivo = pd.Series(
    split_por_indice.loc[ident_completa.index].to_numpy(),
    index=ident_completa["Arquivo_norm"].to_numpy(),
)
# reindexa na mesma ordem de `completa` (já sem as 5 linhas com NaN residual)
split_completa = split_por_arquivo.reindex(completa.index)

idx_treino_c = np.where(split_completa.to_numpy() == "treino")[0]
idx_val_c = np.where(split_completa.to_numpy() == "validação")[0]
idx_teste_c = np.where(split_completa.to_numpy() == "teste")[0]

print(f"Completa — treino: {len(idx_treino_c)} ({len(idx_treino_c)/len(completa)*100:.1f}%), "
      f"val: {len(idx_val_c)} ({len(idx_val_c)/len(completa)*100:.1f}%), "
      f"teste: {len(idx_teste_c)} ({len(idx_teste_c)/len(completa)*100:.1f}%)")
print("Algum split faltando?", split_completa.isna().any())

Completa — treino: 8995 (69.9%), val: 1932 (15.0%), teste: 1939 (15.1%)


Algum split faltando? False


In [18]:
# posições (não rótulos) para indexar X_simples, que segue a ordem de ident_recorte
pos_treino_s = ident_recorte.index.get_indexer(idx_treino)
pos_val_s = ident_recorte.index.get_indexer(idx_val)
pos_teste_s = ident_recorte.index.get_indexer(idx_teste)

assert (pos_treino_s >= 0).all() and (pos_val_s >= 0).all() and (pos_teste_s >= 0).all()
print("Posições OK.")

Posições OK.


## Normalização (média zero, variância unitária) — ajustada só no treino

A média e o desvio-padrão usados pra normalizar são calculados **usando
apenas as linhas de treino**, e esses mesmos números (não recalculados) são
aplicados na validação e no teste.

In [19]:
def normaliza(X, pos_treino):
    media = X[pos_treino].mean(axis=0)
    desvio = X[pos_treino].std(axis=0)
    desvio[desvio == 0] = 1.0  # evita divisão por zero em coluna constante (ex.: one-hot)
    return (X - media) / desvio, media, desvio


X_simples_norm, media_simples, desvio_simples = normaliza(X_simples, pos_treino_s)
X_completa_norm, media_completa, desvio_completa = normaliza(X_completa, idx_treino_c)

print("Simples — média do treino após normalizar (deve ser ~0):",
      X_simples_norm[pos_treino_s].mean(axis=0).round(3))
print("Simples — desvio do treino após normalizar (deve ser ~1):",
      X_simples_norm[pos_treino_s].std(axis=0).round(3))
print()
print("Completa — média do treino após normalizar (deve ser ~0, ~0 nas one-hot):",
      X_completa_norm[idx_treino_c].mean(axis=0).round(2))

Simples — média do treino após normalizar (deve ser ~0): [-0. -0.  0.  0.  0.]
Simples — desvio do treino após normalizar (deve ser ~1): [1. 1. 1. 1. 1.]

Completa — média do treino após normalizar (deve ser ~0, ~0 nas one-hot): [ 0.  0. -0.  0.  0.  0. -0. -0. -0. -0. -0.  0. -0.  0. -0. -0. -0. -0.
 -0.  0. -0. -0. -0. -0.  0. -0.  0.  0. -0.]


## Salvando tudo para as próximas etapas

Salva em `dados_processados/` (fora do git — deriva direto do `banco/`
privado). As Etapas A, B, C... carregam esse arquivo em vez de repetir a
limpeza e a divisão, garantindo que todas usem exatamente a mesma divisão.

In [20]:
import os

os.makedirs("../dados_processados", exist_ok=True)

np.savez(
    "../dados_processados/etapa2.npz",
    # representação simples
    X_simples=X_simples_norm,
    y_simples=y_simples,
    colunas_simples=np.array(colunas_simples),
    pos_treino_simples=pos_treino_s,
    pos_val_simples=pos_val_s,
    pos_teste_simples=pos_teste_s,
    media_simples=media_simples,
    desvio_simples=desvio_simples,
    # representação completa
    X_completa=X_completa_norm,
    y_completa=y_completa,
    colunas_completa=np.array(colunas_completa),
    pos_treino_completa=idx_treino_c,
    pos_val_completa=idx_val_c,
    pos_teste_completa=idx_teste_c,
    media_completa=media_completa,
    desvio_completa=desvio_completa,
    # geral
    especies_recorte=np.array(especies_recorte),
)

print("Salvo em dados_processados/etapa2.npz")
print(os.path.getsize("../dados_processados/etapa2.npz") / 1024, "KB")

Salvo em dados_processados/etapa2.npz
3704.54296875 KB


In [21]:
print("Arquivo_norm duplicado em ident_completa, após correção na base:",
      (ident_completa["Arquivo_norm"].value_counts() > 1).sum())

Arquivo_norm duplicado em ident_completa, após correção na base: 0
